# Case 1
```js
throw new Error("Xin chào")
```

---

* Bên trong Error là gì?
```js
err.message = "Xin chào"; // STRING
```

---

* Khi tôi làm:
```js
String(err)
```
---

* JS gọi:
```js
Error.prototype.toString()
// => `${name}: ${message}`
```

---

➡️ Kết quả:
`"Error: Xin chào"`

---

Vì sao `"err": {}` ?
```js
JSON.stringify(new Error("Xin chào")) === "{}"
//Vì: message, stack, name → non-enumerable
```

# CASE 2
```js
throw new Error({ alo: "hi" })
```

---

* Bên trong Error là gì?
```js
err.message = { alo: "hi" }; // OBJECT
```

---

* Khi stringify:
```js
String(err)
```

---

* JS làm::
```js
String(err.message) // vì toString ghép chuỗi
```

- ➡️ Object → "[object Object]"
Nên kết quả là: `"Error: [object Object]"`
- `"err": {}` vẫn thế -> Vì vẫn là Error → JSON.stringify → {}

In [ ]:
# Case 3
`throw { hi: "Xin chào" }` -> ⚠️ Đây KHÔNG phải Error

* err là gì?
```js
  err = {
    hi: "Xin chào"
  }
```

* Khi stringify:
```js
JSON.stringify(err)
```
➡️ Object thường → serialize bình thường
```json
{ "hi": "Xin chào" }
```

- Nhưng: `String(err)` --> ➡️ Object → "[object Object]"

In [ ]:
const { errorCodes, httpCodes } = require("@/configs/constants");
const { AuthError, EmailVerifyError } = require("@/utils/errors");
const isProduction = require("@/utils/isProduction");
const { JsonWebTokenError } = require("jsonwebtoken");
const errorHandler = (err, _, res, next) => {
    if (res.headerSent) return next(err);

    let status;
    // JWT error
    if (err instanceof JsonWebTokenError) {
        if (err.name === "TokenExpiredError") {
            err = "Token expired";
            status = httpCodes.unauthorized;
        } else if (err.name === "JsonWebTokenError") {
            err = "Invalid token";
            status = httpCodes.unauthorized;
        } else {
            err = "Unauthorized";
            status = httpCodes.unauthorized;
        }
    }

    // Hứng lỗi xác thực chung
    if (err instanceof AuthError) {
        err = err.message || "Unauthorized";
        status = err.statusCode || httpCodes.unauthorized;
    }

    // Hứng lỗi verify email
    if (err instanceof EmailVerifyError) {
        err = err.message || "Forbidden";
        status = err.statusCode || httpCodes.forbidden;
    }

    // Conflict
    if (err?.code === errorCodes.conflict) {
        err = "Conflict";
        status = httpCodes.conflict;
    }

    // Exception: Lỗi không xác định
    res.error(
        {
            err: !isProduction() ? err : "Server error.", // JSON.stringify(): Error: enumerable FALSE -> {} || SQL Error:enumerable TRUE -> {...}
            message: !isProduction() ? String(err) : "Server error.", // Error: error.message
        },
        status,
    );
};

module.exports = errorHandler;
